In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# перебор комбинаций К - ransacReprojThreshold

import cv2
import numpy as np
import json
import os
from pathlib import Path
import pandas as pd


DATA_ROOT = Path("/content/drive/MyDrive/VAC_HOLOD/test-task")
PROJECT_ROOT = Path("/content/solution")
SCRIPT_DIR = PROJECT_ROOT
SCRIPT_DIR.mkdir(exist_ok=True)

SPLIT_FILE = DATA_ROOT / "split.json"
OUTPUT_DIR = SCRIPT_DIR

print("1. Загрузка split.json...")
with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split_data = json.load(f)

train_sessions = split_data.get("train", [])
val_sessions = split_data.get("val", [])

print("2. Сбор референсных точек из train...")
ref_src = {"top": [], "bottom": []}
ref_dst = {"top": [], "bottom": []}

for sess_rel in train_sessions:
    sess_dir = DATA_ROOT / sess_rel
    for cam in ["top", "bottom"]:
        coords_path = sess_dir / f"coords_{cam}.json"
        if not coords_path.exists(): continue
        with open(coords_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair in data:
            if cam in pair["file1_path"]:
                src_list = pair["image1_coordinates"]
                dst_list = pair["image2_coordinates"]
            else:
                src_list = pair["image2_coordinates"]
                dst_list = pair["image1_coordinates"]
            for s, d in zip(src_list, dst_list):
                ref_src[cam].append([s["x"], s["y"]])
                ref_dst[cam].append([d["x"], d["y"]])

# Преобразуем в numpy один раз для скорости
ref_src_np = {cam: np.array(ref_src[cam], dtype=np.float32) for cam in ref_src}
ref_dst_np = {cam: np.array(ref_dst[cam], dtype=np.float32) for cam in ref_dst}

def run_validation(K, thresh):
    errors_top, errors_bottom = [], []
    for sess_rel in val_sessions:
        sess_dir = DATA_ROOT / sess_rel
        for cam in ["top", "bottom"]:
            coords_path = sess_dir / f"coords_{cam}.json"
            if not coords_path.exists(): continue
            with open(coords_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            for pair in data:
                if cam in pair["file1_path"]:
                    src_list = pair["image1_coordinates"]
                    dst_list = pair["image2_coordinates"]
                else:
                    src_list = pair["image2_coordinates"]
                    dst_list = pair["image1_coordinates"]

                src_pts = ref_src_np[cam]
                dst_pts = ref_dst_np[cam]

                for s, d in zip(src_list, dst_list):
                    query = np.array([s["x"], s["y"]], dtype=np.float32)
                    dists = np.linalg.norm(src_pts - query, axis=1)
                    idx = np.argsort(dists)[:K]
                    p_src, p_dst = src_pts[idx], dst_pts[idx]

                    try:
                        H, status = cv2.findHomography(p_src, p_dst, method=cv2.RANSAC, ransacReprojThreshold=thresh)
                        if H is None or status is None: raise ValueError
                        mask = status.ravel().astype(bool)
                        if mask.sum() < 4: raise ValueError
                        H, _ = cv2.findHomography(p_src[mask], p_dst[mask], cv2.RANSAC)
                        mapped = cv2.perspectiveTransform(p_src[mask].reshape(-1, 1, 2), H)[:, 0]
                        dists_local = np.linalg.norm(p_src[mask] - query, axis=1) + 1e-6
                        weights = 1.0 / dists_local
                        weights /= weights.sum()
                        bias = np.sum(weights[:, None] * (p_dst[mask] - mapped), axis=0)
                        pred = cv2.perspectiveTransform(np.array([[[query[0], query[1]]]], dtype=np.float32), H)[0][0]
                        px, py = float(pred[0] + bias[0]), float(pred[1] + bias[1])
                    except Exception:
                        fb = query + np.mean(p_dst - p_src, axis=0)
                        px, py = float(fb[0]), float(fb[1])

                    err = np.hypot(px - d["x"], py - d["y"])
                    if cam == "top": errors_top.append(err)
                    else: errors_bottom.append(err)

    med_top = float(np.mean(errors_top)) if errors_top else 0.0
    med_bottom = float(np.mean(errors_bottom)) if errors_bottom else 0.0
    return med_top, med_bottom

# перебираем 100 вариантов (10 для К и 10 для ransacReprojThreshold)
print("3. Запуск перебора параметров (100 комбинаций)...")
print("Это может занять 5-15 минут в зависимости от CPU.\n")

results = []
total = 100
count = 0

for K in range(1, 11):
    for thr in range(1, 11):
        thr_val = float(thr)
        m_top, m_bot = run_validation(K, thr_val)
        results.append({"K": K, "thresh": thr_val, "top_med": m_top, "bot_med": m_bot, "overall": (m_top+m_bot)/2})
        count += 1
        print(f"[{count}/{total}] K={K:2d} | Thresh={thr_val:4.1f} -> Top: {m_top:6.2f} | Bot: {m_bot:6.2f}")

df = pd.DataFrame(results).sort_values("overall")

print("\n" + "="*60)
print("ТОП-5 ЛУЧШИХ КОМБИНАЦИЙ:")
print(df.head().to_string(index=False))
print("="*60)

# Сохранить лучшую метрику
best = df.iloc[0]
best_metrics = {
    "top_to_door2_med": round(best["top_med"], 2),
    "bottom_to_door2_med": round(best["bot_med"], 2),
    "overall_med": round(best["overall"], 2)
}
Path(OUTPUT_DIR).mkdir(exist_ok=True)
with open(OUTPUT_DIR / "metrics_best.json", "w", encoding="utf-8") as f:
    json.dump(best_metrics, f, indent=2, ensure_ascii=False)

print(f"\nЛучший metrics.json сохранен в: {OUTPUT_DIR / 'metrics_best.json'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Загрузка split.json...
2. Сбор референсных точек из train...
3. Запуск перебора параметров (100 комбинаций)...
Это может занять 5-15 минут в зависимости от CPU.

[1/100] K= 1 | Thresh= 1.0 -> Top: 174.77 | Bot: 179.37
[2/100] K= 1 | Thresh= 2.0 -> Top: 174.77 | Bot: 179.37
[3/100] K= 1 | Thresh= 3.0 -> Top: 174.77 | Bot: 179.37
[4/100] K= 1 | Thresh= 4.0 -> Top: 174.77 | Bot: 179.37
[5/100] K= 1 | Thresh= 5.0 -> Top: 174.77 | Bot: 179.37
[6/100] K= 1 | Thresh= 6.0 -> Top: 174.77 | Bot: 179.37
[7/100] K= 1 | Thresh= 7.0 -> Top: 174.77 | Bot: 179.37
[8/100] K= 1 | Thresh= 8.0 -> Top: 174.77 | Bot: 179.37
[9/100] K= 1 | Thresh= 9.0 -> Top: 174.77 | Bot: 179.37
[10/100] K= 1 | Thresh=10.0 -> Top: 174.77 | Bot: 179.37
[11/100] K= 2 | Thresh= 1.0 -> Top: 148.78 | Bot: 164.96
[12/100] K= 2 | Thresh= 2.0 -> Top: 148.78 | Bot: 164.96
[13/100] K= 2 | Thresh= 3.0 -> 